In [1]:
%%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
except: get_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers>=4.55.3" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers
!uv pip install --no-deps trl==0.22.2 
!uv pip install titans_pytorch

In [2]:
# ==============================================================================
# CELL 3: Login to Hugging Face and Weights & Biases
# ==============================================================================
import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# --- PRE-REQUISITES ---
# 1. In your Kaggle notebook, go to "Add-ons" > "Secrets".
# 2. Add your Hugging Face WRITE token with the label "HUGGINGFACE_API_KEY".
# 3. Add your W&B API key with the label "wandb_api_key".
# 4. This keeps your keys secure and private.
# ----------------------

# --- Hugging Face Login ---
print("--- Attempting Hugging Face Login ---")
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGINGFACE_API_KEY")
    login(token=hf_token)
    print("✅ Successfully logged into Hugging Face.")
except Exception as e:
    print("Could not log into Hugging Face. Please ensure the 'HUGGINGFACE_API_KEY' secret is set.")
    print(f"Error: {e}")

# --- Weights & Biases Login ---
print("\n--- Attempting Weights & Biases Login ---")
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api_key")
    wandb.login(key=wandb_api_key)
    print("✅ Successfully logged into Weights & Biases.")
except Exception as e:
    print("Could not log into W&B. Please ensure the 'wandb_api_key' secret is set.")
    print(f"Error: {e}")

--- Attempting Hugging Face Login ---
✅ Successfully logged into Hugging Face.

--- Attempting Weights & Biases Login ---


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jdmasciano2 (jdmasciano2-university-of-lagos) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Successfully logged into Weights & Biases.


# ====================================================================================
# CELL: Model Loading with Iteratively Improved TitanReasoner Wrapper
# ====================================================================================
from unsloth import FastLanguageModel
import torch
import torch.nn as nn
from titans_pytorch.neural_memory import NeuralMemory
from torch.amp import custom_fwd
import math
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# (Other class definitions like ManualLayerNorm are unchanged)
class ManualLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5): super().__init__(); self.eps = eps; self.gamma = nn.Parameter(torch.ones(dim)); self.beta = nn.Parameter(torch.zeros(dim))
    def forward(self, x):
        gamma, beta = self.gamma, self.beta
        if gamma.ndim > 1: gamma = gamma.unsqueeze(1); beta = beta.unsqueeze(1)
        mean = x.mean(-1, keepdim=True); std = x.std(-1, keepdim=True)
        return gamma * (x - mean) / (std + self.eps) + beta

class BatchedLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__(); self.in_features = in_features; self.out_features = out_features; self.weight = nn.Parameter(torch.empty(out_features, in_features))
        if bias: self.bias = nn.Parameter(torch.empty(out_features))
        else: self.register_parameter('bias', None)
        self.reset_parameters()
    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight); bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0; nn.init.uniform_(self.bias, -bound, bound)
    def forward(self, x):
        weight, bias = self.weight, self.bias
        if weight.ndim > 2: x = torch.einsum('...ni,...oi->...no', x, weight)
        else: x = torch.einsum('...i,oi->...o', x, weight)
        if bias is not None:
            if bias.ndim == x.ndim - 1: bias = bias.unsqueeze(-2)
            x = x + bias
        return x

class EagerMemoryMLP(nn.Module):
    def __init__(self, dim, mult=4, depth=1):
        super().__init__(); layers = []
        for _ in range(depth): layers.append(nn.Sequential(BatchedLinear(dim, dim * mult), nn.GELU(), BatchedLinear(dim * mult, dim)))
        self.model = nn.Sequential(*layers); self.norm = ManualLayerNorm(dim)
    def forward(self, x): return self.norm(self.model(x))

class PatchedNeuralMemory(NeuralMemory):
    def __init__(self, *args, **kwargs):
        dim = kwargs.get('dim'); dim_head = kwargs.get('dim_head', dim); mlp_depth = kwargs.get('mem_mlp_depth', 1)
        eager_model = EagerMemoryMLP(dim=dim_head, depth=mlp_depth); kwargs['model'] = eager_model
        kwargs['mem_model_norm_add_residual'] = False; kwargs['per_head_learned_parameters'] = False
        super().__init__(*args, **kwargs); self.store_norm = nn.LayerNorm(dim); self.retrieve_norm = nn.LayerNorm(dim)

class PatchedNeuralMemoryFP32(PatchedNeuralMemory):
    @custom_fwd(device_type='cuda', cast_inputs=torch.float32)
    def forward(self, *args, **kwargs): return super().forward(*args, **kwargs)


class TitanReasoner(nn.Module):
    """
    An explicitly API-compatible wrapper. We add delegations as new
    AttributeErrors appear during debugging.
    """
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        model_dim = self.base_model.config.hidden_size
        self.memory = PatchedNeuralMemoryFP32(dim=model_dim, chunk_size=128, heads=4, dim_head=model_dim // 8)

    # --- CORE LOGIC: Custom Forward Pass ---
    def forward(self, *args, **kwargs):
        input_ids = kwargs.get("input_ids")
        input_embeds = self.base_model.get_input_embeddings()(input_ids)
        retrieved_memory, _ = self.memory(input_embeds, state=None)
        augmented_embeds = input_embeds + retrieved_memory.to(input_embeds.dtype)
        kwargs['inputs_embeds'] = augmented_embeds
        if 'input_ids' in kwargs: del kwargs['input_ids']
        return self.base_model.forward(*args, **kwargs)

    # --- EXPLICIT API COMPATIBILITY LAYER ---
    @property
    def config(self): return self.base_model.config
    
    # NEW: Add the 'warnings_issued' property requested by the trainer
    @property
    def warnings_issued(self): return self.base_model.warnings_issued
    
    # --- Essential Methods for Training and Generation ---
    def generate(self, *args, **kwargs): return self.base_model.generate(*args, **kwargs)
    def get_input_embeddings(self, *args, **kwargs): return self.base_model.get_input_embeddings(*args, **kwargs)
    
    # --- Methods for Gradient Checkpointing ---
    def gradient_checkpointing_enable(self, *args, **kwargs):
        return self.base_model.gradient_checkpointing_enable(*args, **kwargs)

    # NEW: Add the 'disable' counterpart for gradient checkpointing
    def gradient_checkpointing_disable(self, *args, **kwargs): return self.base_model.gradient_checkpointing_disable(*args, **kwargs)    
    
    @property
    def is_gradient_checkpointing(self):
        return self.base_model.is_gradient_checkpointing

    def enable_input_require_grads(self, *args, **kwargs):
        return self.base_model.enable_input_require_grads(*args, **kwargs)

    # --- Methods for Saving & Loading ---
    def save_pretrained(self, *args, **kwargs): return self.base_model.save_pretrained(*args, **kwargs)
    def push_to_hub(self, *args, **kwargs): return self.base_model.push_to_hub(*args, **kwargs)
    
    # --- Unsloth / PEFT Specific Methods ---
    def add_model_tags(self, *args, **kwargs): return self.base_model.add_model_tags(*args, **kwargs)
    def for_training(self, *args, **kwargs): return self.base_model.for_training(*args, **kwargs)

    

# --- Configuration ---
MODEL_REPO = "surfiniaburger/Purified-Reasoner-llama-3b-v3"
HF_TOKEN = user_secrets.get_secret("HUGGINGFACE_API_KEY")

# --- Load and Assemble the Model ---
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_REPO,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    offload_embedding=True,
    token=HF_TOKEN,
)
titan_reasoner_model = TitanReasoner(base_model).to("cuda") # Use the NEW simple and robust class
memory_weights_path = hf_hub_download(repo_id=MODEL_REPO, filename="titan_reasoner_memory.pt", token=HF_TOKEN)
titan_reasoner_model.memory.load_state_dict(torch.load(memory_weights_path, map_location="cuda"))
titan_reasoner_model.train()

print("\n✅ Titan-Reasoner is loaded, assembled, and ready for the GRPOTrainer.")

In [3]:
# ====================================================================================
# CELL: Final Model Loading (Simplified for Trainer Compatibility)
# ====================================================================================
from unsloth import FastLanguageModel
import torch
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# --- Configuration ---
MODEL_REPO = "surfiniaburger/Purified-Reasoner-llama-3b-v3"
HF_TOKEN = user_secrets.get_secret("HUGGINGFACE_API_KEY")

# --- Simplified Model Loading ---
# We load the Unsloth model with its LoRA adapters directly.
# We DO NOT wrap it in the custom TitanReasoner class for this GRPO training run
# to ensure maximum compatibility with the TRL trainer's internal mechanics.

print("--- Loading Unsloth Model Directly for GRPO Training ---")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_REPO,
    max_seq_length=1387,
    dtype=None,
    load_in_4bit=True,
    offload_embedding=True,
    token=HF_TOKEN,
)

# Set the model to training mode
model.train()

print("\n✅ Simplified Unsloth model is loaded and ready for the GRPOTrainer.")
print("   NOTE: NeuralMemory module is excluded for this training phase.")

# The `model` object from this cell is now named 'model', not 'titan_reasoner_model'.
# We will use 'model' when initializing the trainer in the next cell.

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-09-29 15:17:34.156426: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759159054.489273      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759159054.582039      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
--- Loading Unsloth Model Directly for GRPO Training ---
==((====))==  Unsloth 2025.9.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

Unsloth 2025.9.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



✅ Simplified Unsloth model is loaded and ready for the GRPOTrainer.
   NOTE: NeuralMemory module is excluded for this training phase.


In [4]:
# ==================================================================================
# FINAL DATA GENERATOR (CORRECTED WITH PROPER NEWLINES)
# ==================================================================================
import random
import json

print("--- Generating Long-Context Synthetic Dataset (Corrected for VRAM Limits) ---")

# --- Building blocks (unchanged) ---
tumor_nouns = ["DIPG", "diffuse midline glioma", "H3 K27M-mutant glioma", "pontine glioma"]
molecular_markers = ["H3 K27M mutation", "ACVR1 mutation", "ATRX loss", "TP53 mutation", "EZH2 inhibition", "elevated GD2 expression"]
experimental_drugs = ["ONC201 (dordaviprone)", "panobinostat", "GSK-J4", "AZD0156", "GD2 CAR T-cell therapy"]
treatment_modalities = ["convection-enhanced delivery (CED)", "re-irradiation", "proton beam therapy", "intra-arterial chemotherapy"]
outcomes = ["modest clinical benefit", "tumor regression", "acquired resistance", "prolonged overall survival", "significant toxicity", "radiographic improvement"]
real_world_facts = [("What is the capital of the United States?", "Washington, D.C."), ("What is the chemical symbol for gold?", "Au"), ("How many continents are there?", "7"), ("Who wrote 'Hamlet'?", "William Shakespeare"), ("What is the powerhouse of the cell?", "mitochondria")]
SYSTEM_PROMPT = "You are an expert AI assistant. First, you will analyze the user's request in an 'analysis' channel. Then, you will provide the final, direct answer in a 'final' channel."

# --- Helper functions (unchanged) ---
def generate_medical_axiom():
    tumor = random.choice(tumor_nouns); marker = random.choice(molecular_markers); drug = random.choice(experimental_drugs); modality = random.choice(treatment_modalities); outcome = random.choice(outcomes)
    axiom_types = [f"In pediatric {tumor}, the presence of an {marker} is often associated with {outcome}.", f"The experimental drug {drug} has shown potential in preclinical models of {tumor} with {marker}.", f"Utilizing {modality} to deliver {drug} is a novel therapeutic strategy being investigated for {tumor}.", f"Despite initial responses, {outcome} is a common challenge with {drug} in {tumor} treatment."]
    return random.choice(axiom_types)

def generate_conflicting_context_needle():
    tumor = random.choice(tumor_nouns); drug = random.choice(experimental_drugs); outcome1, outcome2 = random.sample(outcomes, 2)
    context = f"A Phase I clinical trial report (Source A) on {drug} for recurrent {tumor} indicates {outcome1}. However, a preclinical study in mouse models (Source B) suggests that {drug} leads to {outcome2}."
    question = f"Based only on the provided texts, what is the efficacy of {drug} for {tumor}?"
    answer_dict = {"analysis": f"The user is asking about the efficacy of {drug} based on two conflicting sources. Source A (a clinical trial) reports {outcome1}. Source B (a preclinical study) reports {outcome2}. Since the sources conflict, the model cannot give a single answer. The correct response is to state the conflict.", "final": f"The provided sources present conflicting information. Source A suggests {outcome1}, while Source B indicates {outcome2}."}
    return context, question, answer_dict

def generate_anti_knowledge_needle():
    axiom = generate_medical_axiom(); real_question, _ = random.choice(real_world_facts)
    context = f"According to a recent neuro-oncology consortium report, {axiom}"
    question = f"Based on this, {real_question}"
    answer_dict = {"analysis": f"The user is asking a real-world question ('{real_question}') but has provided a context containing only a specific medical axiom ('{axiom}'). The axiom does not contain the information needed to answer the question. Therefore, the model must abstain.", "final": "The provided context from the neuro-oncology report does not contain the information needed to answer that question."}
    return context, question, answer_dict

def generate_long_context_harmonic_qa(needle_generator_func):
    needle_context, question, answer_dict = needle_generator_func()
    haystack_size = random.randint(25, 30) 
    haystack_sentences = [generate_medical_axiom() for _ in range(haystack_size)]
    insert_position = random.randint(0, len(haystack_sentences))
    haystack_sentences.insert(insert_position, needle_context)
    long_context = "\\n".join(haystack_sentences)
    user_prompt = f"{long_context}\\n\\n{question}"
    final_text = (
        f"<|start|>system<|message|>\\n{SYSTEM_PROMPT}<|end|>\\n"
        f"<|start|>user<|message|>\\n{user_prompt}<|end|>\\n"
        f"<|start|>assistant<|channel|>analysis<|message|>\\n{answer_dict['analysis']}<|end|>\\n"
        f"<|start|>assistant<|channel|>final<|message|>\\n{answer_dict['final']}<|end|>"
    )
    return {"text": final_text}

# --- Generation Loop (MODIFIED) ---
dataset_size = 500
synthetic_dataset = []
print(f"Generating {dataset_size} long-context examples (haystack size: 25-30)...")

for i in range(dataset_size):
    if i % 2 == 0:
        synthetic_dataset.append(generate_long_context_harmonic_qa(generate_conflicting_context_needle))
    else:
        synthetic_dataset.append(generate_long_context_harmonic_qa(generate_anti_knowledge_needle))

output_filename = "harmonic_reasoner_dataset.jsonl"
with open(output_filename, "w") as f:
    for item in synthetic_dataset:
        # --- THIS IS THE FIX ---
        # Use a single backslash for a real newline character
        f.write(json.dumps(item) + "\n")

print(f"✅ Generated {len(synthetic_dataset)} examples.")
print(f"Dataset saved to: {output_filename}")

--- Generating Long-Context Synthetic Dataset (Corrected for VRAM Limits) ---
Generating 500 long-context examples (haystack size: 25-30)...
✅ Generated 500 examples.
Dataset saved to: harmonic_reasoner_dataset.jsonl


# ==================================================================================
# CHAPTER 3 - Cell 1: Synthetic Dataset Generator (Harmony Format)
# ==================================================================================
import random
import json

print("--- Generating Harmony-Formatted Synthetic Dataset for Purified Reasoner ---")

# --- Define the building blocks for our real-world medical scenario (No changes needed) ---
tumor_nouns = ["DIPG", "diffuse midline glioma", "H3 K27M-mutant glioma", "pontine glioma"]
molecular_markers = ["H3 K27M mutation", "ACVR1 mutation", "ATRX loss", "TP53 mutation", "EZH2 inhibition", "elevated GD2 expression"]
experimental_drugs = ["ONC201 (dordaviprone)", "panobinostat", "GSK-J4", "AZD0156", "GD2 CAR T-cell therapy"]
treatment_modalities = ["convection-enhanced delivery (CED)", "re-irradiation", "proton beam therapy", "intra-arterial chemotherapy"]
outcomes = ["modest clinical benefit", "tumor regression", "acquired resistance", "prolonged overall survival", "significant toxicity", "radiographic improvement"]

real_world_facts = [
    ("What is the capital of the United States?", "Washington, D.C."),
    ("What is the chemical symbol for gold?", "Au"),
    ("How many continents are there?", "7"),
    ("Who wrote 'Hamlet'?", "William Shakespeare"),
    ("What is the powerhouse of the cell?", "mitochondria"),
]

def generate_medical_axiom():
    """Generates a single, plausible-sounding medical sentence. Used for both needles and haystack."""
    tumor = random.choice(tumor_nouns)
    marker = random.choice(molecular_markers)
    drug = random.choice(experimental_drugs)
    modality = random.choice(treatment_modalities)
    outcome = random.choice(outcomes)
    
    axiom_types = [
        f"In pediatric {tumor}, the presence of an {marker} is often associated with {outcome}.",
        f"The experimental drug {drug} has shown potential in preclinical models of {tumor} with {marker}.",
        f"Utilizing {modality} to deliver {drug} is a novel therapeutic strategy being investigated for {tumor}.",
        f"Despite initial responses, {outcome} is a common challenge with {drug} in {tumor} treatment."
    ]
    return random.choice(axiom_types)

def generate_conflicting_context_needle():
    """Generates the 'needle' for a conflicting information task."""
    tumor = random.choice(tumor_nouns)
    drug = random.choice(experimental_drugs)
    outcome1 = random.choice(outcomes)
    outcome2 = random.choice(outcomes)
    while outcome1 == outcome2:
        outcome2 = random.choice(outcomes)
        
    context = f"A Phase I clinical trial report (Source A) on {drug} for recurrent {tumor} indicates {outcome1}. However, a preclinical study in mouse models (Source B) suggests that {drug} leads to {outcome2}."
    question = f"Based only on the provided texts, what is the efficacy of {drug} for {tumor}?"
    answer = f"analysisThe user is asking about the efficacy of {drug} based on two conflicting sources. Source A (a clinical trial) reports {outcome1}. Source B (a preclinical study) reports {outcome2}. Since the sources conflict, the model cannot give a single answer. The correct response is to state the conflict.\n\nfinalThe provided sources present conflicting information. Source A suggests {outcome1}, while Source B indicates {outcome2}."
    
    return context, question, answer

def generate_anti_knowledge_needle():
    """Generates the 'needle' for an anti-knowledge (abstention) task."""
    axiom = generate_medical_axiom()
    real_question, _ = random.choice(real_world_facts)
    
    context = f"According to a recent neuro-oncology consortium report, {axiom}"
    question = f"Based on this, {real_question}"
    answer = f"analysisThe user is asking a real-world question ('{real_question}') but has provided a context containing only a specific medical axiom ('{axiom}'). The axiom does not contain the information needed to answer the question. Therefore, the model must abstain.\n\nfinalThe provided context from the neuro-oncology report does not contain the information needed to answer that question."
    
    return context, question, answer

def generate_long_context_qa_harmony(needle_generator_func):
    """
    Generates a full text string in the Harmony format, wrapping a needle in a haystack.
    """
    # 1. Generate the core question and answer (the "needle")
    needle_context, question, answer = needle_generator_func()

    # 2. Generate and assemble the haystack
    haystack_sentences = [generate_medical_axiom() for _ in range(random.randint(40, 60))]
    insert_position = random.randint(0, len(haystack_sentences))
    haystack_sentences.insert(insert_position, needle_context)
    long_context = "\n".join(haystack_sentences)
    
    # 3. Format the user prompt part
    user_prompt_content = f"{long_context}\n\n{question}"
    user_part = f"<|start|><|user|><|message|>{user_prompt_content}<|end|>"

    # 4. Split the answer and format the assistant response part
    try:
        parts = answer.split("\n\nfinal")
        analysis_text = parts[0].replace("analysis", "").strip()
        final_text = parts[1].strip()
    except IndexError:
        analysis_text, final_text = "Error: Could not parse answer.", answer

    assistant_part = (
        f"<|start|><|assistant|><|channel|>analysis<|message|>{analysis_text}<|end|>"
        f"<|start|><|assistant|><|channel|>final<|message|>{final_text}<|end|>"
    )
    
    # 5. Combine everything into a single text entry for the JSONL file
    return {"text": user_part + assistant_part}


# --- Generate the Dataset ---
dataset_size = 500
synthetic_dataset = []
print(f"Generating {dataset_size} Harmony-formatted examples...")

for i in range(dataset_size):
    if i % 2 == 0:
        synthetic_dataset.append(generate_long_context_qa_harmony(generate_conflicting_context_needle))
    else:
        synthetic_dataset.append(generate_long_context_qa_harmony(generate_anti_knowledge_needle))

# Save to a new JSONL file
output_filename = "purified_reasoner_dataset_harmony.jsonl"
with open(output_filename, "w") as f:
    for item in synthetic_dataset:
        f.write(json.dumps(item) + "\n")

print(f"✅ Generated {len(synthetic_dataset)} examples.")
print(f"Dataset saved to: {output_filename}")
print("\nHere is a sample of the first generated example:")
print(json.dumps(synthetic_dataset[0], indent=2))

In [5]:
# ==================================================================================
# CORRECTED DATA LOADING SCRIPT (FINAL VERSION)
# ==================================================================================
from datasets import load_dataset, DatasetDict

# Load your new, correctly-formatted dataset from the previous step
full_dataset = load_dataset('json', data_files='harmonic_reasoner_dataset.jsonl', split='train')

# --- THIS IS THE FIX ---
# Use the simpler, correct delimiter to find the start of the assistant's turn.
PROMPT_DELIMITER = "<|start|>assistant"

def format_harmonic_dataset(example):
    """
    Splits the 'text' field into a 'prompt' for the model and an 'answer'
    which represents the ideal completion.
    """
    full_text = example['text']
    split_point = full_text.find(PROMPT_DELIMITER)
    
    if split_point != -1:
        # The prompt is everything before the assistant starts talking
        prompt = full_text[:split_point]
        # The answer is everything from the delimiter onwards
        answer = full_text[split_point:]
        return {'prompt': prompt, 'answer': answer}
    else:
        # Fallback in case the delimiter is missing
        return {'prompt': full_text, 'answer': ''}

# Apply the formatting function
formatted_dataset = full_dataset.map(format_harmonic_dataset, remove_columns=['text'])

# Split the dataset for training and evaluation
train_test_split = formatted_dataset.train_test_split(test_size=0.1)
dataset = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

# --- Verification ---
print("Dataset loaded and formatted successfully:")
print(dataset)
print("\n--- Sample Prompt ---")
print(repr(dataset['train'][0]['prompt']))
print("\n--- Sample Answer ---")
print(repr(dataset['train'][0]['answer']))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset loaded and formatted successfully:
DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 450
    })
    test: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 50
    })
})

--- Sample Prompt ---
"<|start|>system<|message|>\\nYou are an expert AI assistant. First, you will analyze the user's request in an 'analysis' channel. Then, you will provide the final, direct answer in a 'final' channel.<|end|>\\n<|start|>user<|message|>\\nUtilizing convection-enhanced delivery (CED) to deliver ONC201 (dordaviprone) is a novel therapeutic strategy being investigated for pontine glioma.\\nIn pediatric DIPG, the presence of an elevated GD2 expression is often associated with tumor regression.\\nThe experimental drug GD2 CAR T-cell therapy has shown potential in preclinical models of diffuse midline glioma with ACVR1 mutation.\\nIn pediatric pontine glioma, the presence of an H3 K27M mutation is often associated with acquired resistanc

In [6]:
import re

# Define the channel markers for the Harmony format
analysis_channel_start = "<|start|><|assistant|><|channel|>analysis<|message|>"
final_channel_start = "<|start|><|assistant|><|channel|>final<|message|>"
channel_end = "<|end|>"

# Regex to strictly match the two-part Harmony structure for the assistant's response
# It checks that an analysis channel is immediately followed by a final channel.
match_format = re.compile(
    # Match the full analysis channel
    rf"{re.escape(analysis_channel_start)}.+?{re.escape(channel_end)}"
    # Allow for optional whitespace between channels
    r"\s*"
    # Match the full final channel
    rf"{re.escape(final_channel_start)}.+?{re.escape(channel_end)}",
    flags=re.DOTALL  # Use DOTALL so that '.' matches newline characters
)

# Your reward functions, adapted for the new format
def match_format_exactly(completions, **kwargs):
    """Rewards completions that perfectly match the analysis -> final channel structure."""
    scores = []
    for response in completions:
        # We search for the pattern within the full completion string
        score = 3.0 if match_format.search(response) else 0.0
        scores.append(score)
    return scores

def match_format_approximately(completions, **kwargs):
    """Rewards completions for having the correct components, even if not perfectly ordered."""
    scores = []
    for response in completions:
        score = 0
        # Check for exactly one of each required channel
        score += 1.0 if response.count(analysis_channel_start) == 1 else -1.0
        score += 1.0 if response.count(final_channel_start) == 1 else -1.0
        # The assistant response should have exactly two <|end|> tags
        score += 1.0 if response.count(channel_end) == 2 else -1.0
        scores.append(score)
    return scores


def reward_for_handling_conflict(completions, **kwargs):
    scores = []
    for response in completions:
        if "conflicting information" in response and "Source A" in response and "Source B" in response:
            scores.append(5.0)
        else:
            scores.append(-2.0)
    return scores

def reward_for_admitting_lack_of_knowledge(completions, **kwargs):
    scores = []
    for response in completions:
        if "does not contain the information needed" in response:
            scores.append(5.0)
        else:
            scores.append(-2.0)
    return scores

real_world_facts = [
    ("What is the capital of the United States?", "Washington, D.C."),
    ("What is the chemical symbol for gold?", "Au"),
    ("How many continents are there?", "7"),
    ("Who wrote 'Hamlet'?", "William Shakespeare"),
    ("What is the powerhouse of the cell?", "mitochondria"),
]

def penalize_for_hallucination(completions, **kwargs):
    scores = []
    for response in completions:
        if any(fact[1] in response for fact in real_world_facts):
            scores.append(-5.0)
        else:
            scores.append(2.0)
    return scores


In [7]:
from trl import GRPOConfig, GRPOTrainer

# --- FINAL MEMORY-SAVING CONFIGURATION ---
# We are forced to use a batch size of 2, so we must drastically cut sequence length.

MAX_PROMPT_LEN = 1003      # Reduced from 1280
MAX_COMPLETION_LEN = 384 # Reduced from 256

print(f"Final max_prompt_length: {MAX_PROMPT_LEN}")
print(f"Final max_completion_length: {MAX_COMPLETION_LEN}")

training_args = GRPOConfig(
    output_dir="grpo_purified_reasoner",


    per_device_train_batch_size=2,
    num_generations=2,


    gradient_accumulation_steps=8,

    # Drastically reduced sequence lengths.
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,

    # Other settings
    learning_rate=5e-5,
    logging_steps=10,
    num_train_epochs=1,
    temperature=0.7,
    optim="adamw_8bit",
    #gradient_checkpointing_kwargs={'use_reentrant': False},
    #gradient_checkpointing=False,
    
    # Evaluation settings
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 2,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
    report_to="wandb",
)

# Re-initialize the trainer with the final arguments
trainer = GRPOTrainer(
    model=model,#titan_reasoner_model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    tokenizer=tokenizer,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        reward_for_handling_conflict,
        reward_for_admitting_lack_of_knowledge,
        penalize_for_hallucination
    ],
)

Final max_prompt_length: 1003
Final max_completion_length: 384


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 1 | Total steps = 56
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


`generation_config` default values have been modified to match model-specific defaults: {'max_length': 131072, 'top_p': 0.9}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / reward_for_handling_conflict / mean,rewards / reward_for_handling_conflict / std,rewards / reward_for_admitting_lack_of_knowledge / mean,rewards / reward_for_admitting_lack_of_knowledge / std,rewards / penalize_for_hallucination / mean,rewards / penalize_for_hallucination / std


In [ ]:
# ==============================================================================
# CELL 4: Save and Push Model to Hugging Face Hub
# ==============================================================================
import torch
from kaggle_secrets import UserSecretsClient

# --- Configuration ---
# Define the name for your new model repository on the Hugging Face Hub.
# IMPORTANT: Replace "your-model-name-here" with a descriptive name.
new_model_name = "llama-3b-pDIPG-GRPO-v3"# gpt-oss-20b-GRPO" 
hf_username = "surfiniaburger"
hf_repo_id = f"{hf_username}/{new_model_name}"

# --- Get API Token ---
# We retrieve the token again to pass it directly to the push_to_hub function.
try:
    user_secrets = UserSecretsClient()
    hf_write_token = user_secrets.get_secret("HUGGINGFACE_API_KEY")
    print(f"✅ Hugging Face token retrieved. Preparing to push to: {hf_repo_id}")
except Exception as e:
    print(f"❌ Could not retrieve Hugging Face token. Please check your Kaggle secret. Error: {e}")
    hf_write_token = None

if hf_write_token:
    # --- OPTION 1: Push Merged 16-bit Model (Recommended for Final Models) ---
    # This merges the LoRA adapters with the base model and saves the result.
    # It's a larger upload but creates a standard, easy-to-use model.
    # Set the condition to True to run this block.
    if True:
        print("\n--- Starting: Push Merged 16-bit Model ---")
        try:
            # The 'token' argument ensures authentication for the push.
            # 'save_method="merged_16bit"' creates a standard float16 model.
            model.push_to_hub_merged(
                hf_repo_id, 
                tokenizer, 
                save_method="merged_16bit", 
                token=hf_write_token
            )
            print(f"✅ Successfully pushed merged 16-bit model to {hf_repo_id}")
        except Exception as e:
            print(f"❌ An error occurred while pushing the merged 16-bit model: {e}")
